
# Locus Zoom
This notebook builds an interactive Manhattan plot dashboard for GWAS analysis. It loads configuration settings from a YAML file, processes genomic data using PySpark and pandas, and visualizes significant SNPs across selected traits and chromosomes. The dashboard allows users to dynamically explore results, highlighting genomic positions and associated tag loci, with interactive plotting features for deeper inspection.

In [0]:
%pip install mplcursors
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyspark.sql import functions as F
import yaml
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mplcursors


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ============================================================
# MANHATTAN DASHBOARD (CONFIG-DRIVEN)
# Trait + Chrom selector
# Shows ONLY significant SNPs
# Labels: genomic position + taglo IDs
# ============================================================



# -----------------------------
# LOAD CONFIG (SINGLE SOURCE)
# -----------------------------
CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

# paths / data
GWAS_TABLE = CONFIG["data"]["gwas_table"]

# analysis parameters (safe fallback)
P_THRESHOLD = (
    CONFIG.get("analysis", {})
          .get("manhattan_p_threshold", 1e-6)
)

print("GWAS table :", GWAS_TABLE)
print("P-threshold:", P_THRESHOLD)

# -----------------------------
# LOAD GWAS DATA (ONCE)
# -----------------------------
df_spark = (
    spark.table(GWAS_TABLE)
    .select(
        "trait",
        "chrom",
        "start",
        "p_wald",
        "beta",
        "taglo_id1",
        "taglo_id2",
        "taglo_id3",
        "taglo_id4"
    )
    .filter(F.col("p_wald") < P_THRESHOLD)
)

# -----------------------------
# MELT TAGLO COLUMNS
# -----------------------------
df_long = (
    df_spark
    .selectExpr(
        "trait",
        "chrom",
        "start",
        "p_wald",
        "beta",
        "stack(4, "
        "'t1', taglo_id1, "
        "'t2', taglo_id2, "
        "'t3', taglo_id3, "
        "'t4', taglo_id4"
        ") as (copy, taglo_id)"
    )
    .filter(F.col("taglo_id").isNotNull())
)

df = df_long.toPandas()

# -----------------------------
# CLEANUP
# -----------------------------
df = df.rename(columns={"start": "pos"})

for c in ["pos", "p_wald", "beta"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["pos", "p_wald"])
df["neglog10p"] = -np.log10(df["p_wald"])

# combine taglo IDs per SNP
df["taglo_id"] = df["taglo_id"].astype(str)
df = (
    df.groupby(
        ["trait", "chrom", "pos", "p_wald", "beta", "neglog10p"]
    )["taglo_id"]
    .apply(lambda x: ",".join(sorted(set(x))))
    .reset_index()
)

print("Rows:", len(df))
print("Traits:", df["trait"].nunique())
print("Chromosomes:", sorted(df["chrom"].unique()))

# -----------------------------
# WIDGETS
# -----------------------------
trait_w = widgets.Dropdown(
    options=sorted(df["trait"].unique()),
    description="Trait:",
    layout=widgets.Layout(width="45%")
)

chrom_w = widgets.Dropdown(
    options=["ALL"] + sorted(df["chrom"].unique()),
    value="ALL",
    description="Chrom:",
    layout=widgets.Layout(width="35%")
)

out = widgets.Output()

display(
    widgets.VBox([
        widgets.HBox([trait_w, chrom_w]),
        out
    ])
)

# -----------------------------
# DASHBOARD FUNCTION
# -----------------------------
def update_plot(*args):
    with out:
        clear_output(wait=True)

        trait = trait_w.value
        chrom = chrom_w.value

        sub = df[df["trait"] == trait].copy()
        if chrom != "ALL":
            sub = sub[sub["chrom"] == chrom]

        if sub.empty:
            print(f"No significant SNPs (p < {P_THRESHOLD})")
            return

        fig, ax = plt.subplots(figsize=(14, 5))

        sc = ax.scatter(
            sub["pos"] / 1e6,
            sub["neglog10p"],
            c=sub["beta"],
            cmap="RdBu_r",
            s=40,
            edgecolor="black",
            linewidth=0.4
        )

        # labels: position + taglo IDs
        for _, r in sub.iterrows():
            ax.text(
                r["pos"] / 1e6,
                r["neglog10p"] + 0.03,
                f"{int(r['pos'])}\n{r['taglo_id']}",
                fontsize=8,
                ha="center",
                va="bottom"
            )

        ax.set_title(
            f"{trait} | {chrom} | p < {P_THRESHOLD}"
        )
        ax.set_xlabel("Genomic position (Mbp)")
        ax.set_ylabel("-log10(p)")

        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label("β (effect size)")

        plt.tight_layout()
        plt.show()

# -----------------------------
# CONNECT + INITIAL CALL
# -----------------------------
trait_w.observe(update_plot, names="value")
chrom_w.observe(update_plot, names="value")

update_plot()


GWAS table : bmqg.gwas.run_local_20251207
P-threshold: 1e-06
Rows: 13342
Traits: 59
Chromosomes: ['ST4.03ch00', 'ST4.03ch01', 'ST4.03ch02', 'ST4.03ch03', 'ST4.03ch04', 'ST4.03ch05', 'ST4.03ch06', 'ST4.03ch07', 'ST4.03ch08', 'ST4.03ch09', 'ST4.03ch10', 'ST4.03ch11', 'ST4.03ch12']
